# DRISHTI — full pipeline on a Colab / Kaggle session

Run the **entire 11-stage pipeline** — `s0_ingest → … → report` — on the hosted machine, then
download the bundle (model + successful deliverables + accuracy report + logs). The session's CPU,
host RAM, and T4 GPU are used as appropriate; your computer only needs to view the results.

> Sibling notebook `drishti_cloud_t4.ipynb` does the tier split and stops after dense reconstruction.
> This notebook runs all stages here. Both use the same content-hashed bundle contract.

It installs the real package and runs the real CLI. Required-input/dependency errors are explicit;
optional fallbacks and skipped exports are recorded. Inspect the actual results before claiming
completion, accuracy, or performance.

### Run order
1. Enable a T4 accelerator and Internet on Colab/Kaggle.
2. Run the initial checkpoint-download permission cell and set up writable source in §1.
3. Run §0–§4 (GPU → code → install → doctor).
4. In §5 pick **one** input option.
5. Run **one** §6 block (6B for aerial samples), then §7–§9 (inspect → logs → download).

## First time on Kaggle?

Follow the **[complete beginner guide](KAGGLE-GUIDE.md)** (`notebooks/KAGGLE-GUIDE.md` in the repository). Keep it open alongside the notebook.

Run cells selectively: **one** input option and **one** reconstruction block. For aerial samples choose **6B**. Do not use Run All on the unedited notebook. The balanced TSDF/Poisson path uses CPU and host RAM.

This version includes the S1 adjacent-frame coverage fix in the source checkout. **Commit and push the source plus notebook before reimporting**; section 1 fetches your repository and checks for the old bug. No manual source-patch cell is needed for a corrected checkout.

The initial setup gives Kaggle paths priority even if `/content` exists. The installer includes `texture` and `export`; the guide explains explicit GLB selection. Option C accepts a bundle directory or full ZIP on either host. Run-cell retries append timestamped attempts to the console log. Resume keeps saved settings and accepts no `--set` overrides.

In [ ]:
# Hosted setup: Kaggle takes precedence even when a /content directory also exists.
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
ON_COLAB = not ON_KAGGLE and Path("/content").is_dir()
HOST_ROOT = Path("/kaggle/working") if ON_KAGGLE else (Path("/content") if ON_COLAB else Path.cwd())
os.environ["DRISHTI__models__allow_download"] = "true"
print("Host:", "Kaggle" if ON_KAGGLE else ("Colab" if ON_COLAB else "local"))
print("Working root:", HOST_ROOT)

## 0 · Confirm the GPU

Confirm the allocated NVIDIA GPU before continuing. The hosted neural stages use CUDA; the balanced
S7 TSDF path and S8 Poisson meshing use CPU and host RAM. Later, `doctor` also checks the installed
Python environment's GPU access.

In [ ]:
!nvidia-smi

## 1 · Get the code

Set your real repository URL and an existing branch/tag below. The clone cell resets an existing
checkout and discards local edits, so avoid rerunning it for a same-session pipeline retry.

On Kaggle, an attached source dataset is read-only. Copy it into `/kaggle/working`, then replace/skip
this clone cell using [the guide's source-snapshot setup](KAGGLE-GUIDE.md#section-1--get-the-source-code).
Do not run the project directly under `/kaggle/input`.

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/suryanarayan100406/rtoos_test.git"   # change for a fork
REPO_REF = "main"                                         # existing branch or tag
REPO_DIR = HOST_ROOT / "drishti"

# Always land on the LATEST commit of REPO_REF. A plain "clone only if missing" silently keeps stale
# code across re-runs in the same session (e.g. Kaggle's /kaggle/working persists) -- that is how an
# already-fixed bug can appear to "come back". If the repo exists we fetch + hard-reset to the remote
# ref; otherwise we clone fresh.
def sh(*cmd, **kw):
    return subprocess.run(cmd, check=True, **kw)

if (REPO_DIR / ".git").exists():
    print(f"{REPO_DIR} present -> syncing to origin/{REPO_REF} (discarding local changes)")
    sh("git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF)
    sh("git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD")
    sh("git", "-C", str(REPO_DIR), "clean", "-fd")
else:
    if REPO_DIR.exists():   # non-git leftovers -> remove so the clone is clean
        import shutil; shutil.rmtree(REPO_DIR)
    sh("git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR))

os.chdir(REPO_DIR)
print("HEAD:", subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
                              capture_output=True, text=True).stdout.strip())

# Catch the known old-checkout S1 failure before installing models or downloading a dataset.
s1_source = (REPO_DIR / "src/drishti/stages/s1_frameqa.py").read_text(encoding="utf-8")
if "zip(accepted, accepted[1:], strict=True)" in s1_source:
    raise RuntimeError("This checkout still has the S1 coverage bug. Push the corrected source to "
                       "REPO_URL / REPO_REF, then rerun this setup cell. Reimporting the notebook alone is not enough.")

# Defaults so §6 never NameErrors even if you run only one import cell in §5.
MISSION = ""            # a dataset descriptor .yaml  -> fresh full run
RESUME_BUNDLE = None    # a partial bundle directory  -> resume on the GPU
print("cwd:", Path.cwd())


## 2 · Install DRISHTI

The cell installs core first, then each capability group separately, including `texture` and `export`. Transformers comes from `depth`, laspy from `export`, and xatlas/Pillow from `texture`. An optional install failure is printed and may still leave a stage blocked; read the output.

For kernels newer than Python 3.12, `uv` creates an isolated **Python 3.12** environment under the selected host root for the Open3D compatibility expected by this notebook. On older kernels installation runs in place. The selected interpreter is `PY`; run cells use it explicitly. Environment creation errors stop the cell.

The final CUDA check must succeed. The module probe actually imports each module and prints `OK` or `MISSING/BROKEN` with the exception. Fix failures and run §4 doctor before starting reconstruction. Hosted versions and available wheels can change.

In [ ]:
# Install into one explicit interpreter; keep all hosted files under HOST_ROOT.
import os, sys, subprocess
from pathlib import Path

os.chdir(REPO_DIR)
USE_VENV = sys.version_info[:2] > (3, 12)
print("Kernel Python:", sys.version.split()[0], "| isolated Python 3.12:", USE_VENV)
if USE_VENV:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    VENV = HOST_ROOT / "venv312"
    subprocess.run(["uv", "python", "install", "3.12"], check=True)
    if not (VENV / "bin/python").exists():
        subprocess.run(["uv", "venv", "--seed", "--python", "3.12", str(VENV)], check=True)
    os.environ["VIRTUAL_ENV"] = str(VENV)
    os.environ["PATH"] = f"{VENV / 'bin'}{os.pathsep}{os.environ['PATH']}"
    PY = str(VENV / "bin/python")
else:
    PY = sys.executable
print("Installing into:", PY)

def pipi(*pkgs, required=False):
    cmd = (["uv", "pip", "install", "--python", PY] if USE_VENV
           else [PY, "-m", "pip", "install", "-q"]) + list(pkgs)
    print("  $", " ".join(cmd))
    result = subprocess.run(cmd)
    if result.returncode:
        if required:
            raise RuntimeError(f"Required install failed: {' '.join(pkgs)}")
        print("Install failed for", pkgs, "— review the error and doctor readiness.")

# Core first (including NumPy), then independent heavy groups.
pipi("-e", ".", required=True)
for group in ("video", "geo", "recon", "poses", "spine", "telem", "depth", "texture", "export", "server"):
    pipi("-e", f".[{group}]")

subprocess.run([PY, "-c",
    "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available()); "
    "assert torch.cuda.is_available(), 'CUDA unavailable in the selected Python environment'"], check=True)
probe = (
    "import importlib\n"
    "mods=['av','cv2','pyproj','rasterio','open3d','trimesh','pycolmap','gtsam','torch','transformers','laspy','xatlas','PIL']\n"
    "for name in mods:\n"
    "    try:\n"
    "        importlib.import_module(name)\n"
    "        print(name, 'OK')\n"
    "    except Exception as exc:\n"
    "        print(name, 'MISSING/BROKEN:', exc)\n"
)
subprocess.run([PY, "-c", probe], check=True)

## 3 · Blender — for the FBX deliverable

FBX is a required output, exported out-of-process through headless **Blender ≥ 4.0**. This cell downloads portable Blender 4.2 LTS under the selected host root and puts it on `PATH`.

Without Blender, FBX is recorded as skipped and S10 degraded. Other formats are attempted independently; inspect their actual export results.

In [ ]:
import os, shutil, subprocess, urllib.request
from pathlib import Path

BASE = HOST_ROOT

def _fetch(url, dest):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "drishti"})
        with urllib.request.urlopen(req, timeout=90) as r:
            if getattr(r, "status", 200) >= 400:
                return False
            with open(dest, "wb") as f:
                shutil.copyfileobj(r, f)
        return dest.stat().st_size > 1_000_000
    except Exception as e:
        print("   ", url.rsplit("/", 1)[-1], "->", e)
        return False

if shutil.which("blender"):
    print("blender already on PATH:", shutil.which("blender"))
else:
    tarxz = BASE / "blender.tar.xz"
    got = None
    for v in ["4.2.5", "4.2.4", "4.2.3", "4.2.1", "4.2.0"]:
        print("trying Blender", v)
        if _fetch(f"https://download.blender.org/release/Blender4.2/blender-{v}-linux-x64.tar.xz", tarxz):
            got = v
            break
    if not got:
        print("\n! Blender not fetched — FBX will be skipped (s10_export degraded). This is non-fatal.")
    else:
        dest = BASE / "blender"
        shutil.rmtree(dest, ignore_errors=True)
        dest.mkdir()
        subprocess.run(["tar", "-xf", str(tarxz), "-C", str(dest), "--strip-components=1"], check=True)
        link = Path("/usr/local/bin/blender")
        if link.exists() or link.is_symlink():
            link.unlink()
        link.symlink_to(dest / "blender")
        print(subprocess.run(["blender", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

## 4 · Doctor — the honest gate (spend GPU minutes only if this is green)

Expect **cuda: yes** and every stage **ready** (all extras + the two gap packages installed; the FBX
tool present if you ran §3). Anything still `blocked` here will fail loudly when reached — fix it first.

In [ ]:
!drishti doctor

## 5 · Provide inputs (IMPORT) — run **ONE** of the import cells below

| Option | Use when | What it sets |
|--------|----------|--------------|
| **E · upload your own data** ⭐ | you have your **own** footage and want it configured + run with **no repo commit** | `MISSION` (auto-generates the descriptor) |
| **D · generate here** | first real run / you have no data of your own | `MISSION` (fresh full run) |
| **A/B · your own mission** | you already hand-wrote a `mission.yaml` descriptor | `MISSION` (fresh full run) |
| **C · resume a partial bundle** | you already ran S0/S1 locally and want to continue | `RESUME_BUNDLE` |

A DRISHTI input is a `mission.yaml` **descriptor** plus the files it points at (video + GPS track, and
any optional IMU / baro / intrinsics / RTK). **Option E builds that descriptor for you from a simple
upload form** — the no-commit path for your own data; the other options follow it. See `docs/GUIDE.md`
for the full descriptor contract.

In [ ]:
# == IMPORT · Option D — GENERATE a realistic dataset on this GPU (no upload) ==================
# Assembles the real MP4+SRT contract from open, GPS-tagged aerial imagery: the pixels + per-frame GPS
# are the source's OWN (real drone photos), only the playback timing is synthesized. A tiny ground-truth
# SYNTHETIC village is also available (its GPS is exact -> accuracy is actually checkable).
# Real aerial sets fly ~100-130 m AGL -> run Block 6B (Kaggle ~30 GB) so the dense stage reaches ground.
DATASET = "waterbury"  # REAL:  waterbury(TOWN - most buildings, 248 imgs -> ~124s, ~1.9GB, use 6B) |
                       #        aukerman(farmstead house+barn ~543MB) | brighton_beach(fast ~62MB) |
                       #        caliterra(terrain + a few structures ~272MB) | lewis(145 imgs ~610MB)
                       # SYNTH: synthetic (small ~19-bldg village; exact GPS -> accuracy checkable)

# Synthetic sets are rendered by make_sample_dataset.py; each entry is that script's flags.
_SYNTH = {
    "synthetic":      ["--synthetic"],
    "synthetic_city": ["--synthetic"],
}
if DATASET in _SYNTH:
    !python scripts/make_sample_dataset.py {" ".join(_SYNTH[DATASET])}
    _name = "synthetic_city" if DATASET in ("synthetic", "synthetic_city") else DATASET
else:
    !python scripts/make_sample_dataset.py --dataset {DATASET}
    _name = DATASET

import os
from pathlib import Path
MISSION = str(REPO_DIR / "configs" / "datasets" / f"{_name}.yaml")
RESUME_BUNDLE = None
os.environ["DRISHTI_MISSION"] = MISSION
assert Path(MISSION).is_file(), MISSION
print("\nMISSION =", MISSION, "\n")
print(Path(MISSION).read_text())

In [ ]:
# == IMPORT · Options A/B — bring YOUR OWN mission (fresh full run) =============================
# Uncomment ONE. Point at the descriptor AND make sure the files it references are present.
from pathlib import Path

# A) Google Drive (best for a large video):
# from google.colab import drive; drive.mount("/content/drive")
# MISSION = "/content/drive/MyDrive/drishti/my_mission.yaml"

# B) Direct upload of small files (descriptor + video + telemetry together):
# from google.colab import files; up = files.upload()
# MISSION = next(k for k in up if k.endswith((".yaml", ".yml")))

RESUME_BUNDLE = None
assert MISSION and Path(MISSION).is_file(), (
    f"MISSION not set/found ({MISSION!r}). Uncomment A or B above, or use Option D.")
print("mission:", MISSION)

In [ ]:
# == IMPORT · Option C — RESUME a partial bundle (Kaggle or Colab) ==========================
RUN_OPTION_C = False
BUNDLE_SOURCE = ""  # writable bundle directory, or a FULL bundle ZIP under /kaggle/input/...
INPUT_ROOT = ""     # folder containing recorded relative inputs; blank = REPO_DIR

if RUN_OPTION_C:
    import json, tempfile, zipfile
    from pathlib import Path

    if not BUNDLE_SOURCE:
        if not ON_COLAB:
            raise ValueError("Set BUNDLE_SOURCE to your bundle folder or uploaded full ZIP.")
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one full bundle ZIP.")
        BUNDLE_SOURCE = archives[0]
    candidate = Path(BUNDLE_SOURCE).expanduser().resolve()
    if candidate.is_file() and zipfile.is_zipfile(candidate):
        restored = Path(tempfile.mkdtemp(prefix="drishti-restored-", dir=HOST_ROOT))
        with zipfile.ZipFile(candidate) as archive:
            archive.extractall(restored)
        manifests = list(restored.rglob("manifest.json"))
        if len(manifests) != 1:
            raise ValueError(f"Expected one bundle manifest in {restored}; found {len(manifests)}.")
        candidate = manifests[0].parent
    if not (candidate / "manifest.json").is_file():
        raise FileNotFoundError(f"No manifest.json in {candidate}")
    if Path("/kaggle/input") in candidate.parents:
        raise ValueError("Copy the bundle to /kaggle/working before resuming; inputs are read-only.")
    source_root = Path(INPUT_ROOT).expanduser().resolve() if INPUT_ROOT else REPO_DIR
    saved = json.loads((candidate / "manifest.json").read_text())
    descriptor = saved.get("dataset_snapshot") or {}
    inputs = saved.get("inputs") or []
    for path in (descriptor.get("video"), (descriptor.get("telemetry") or {}).get("path")):
        if not path or (source_root / path).is_file():
            continue
        recorded = next((item for item in inputs if item.get("path") == path and item.get("present")), None)
        if recorded and (candidate / path).is_file() and __import__("drishti.bundle.hashing", fromlist=["sha256_file"]).sha256_file(candidate / path) == recorded.get("sha256"):
            (source_root / path).parent.mkdir(parents=True, exist_ok=True)
            import shutil
            shutil.copy2(candidate / path, source_root / path)
            continue
        raise FileNotFoundError(f"Restore original input {path!r}; relative paths use {source_root}")
    os.chdir(source_root)
    RESUME_BUNDLE = BUNDLE = str(candidate)
    MISSION = ""
    print("Will resume:", RESUME_BUNDLE, "| input working directory:", Path.cwd())
    print("Run 6B next. Saved configuration is reused; completed fresh stages are skipped.")
else:
    print("Option C disabled. Use one fresh-input option instead.")

### Option E — UPLOAD YOUR OWN DATA (no repo commit) ⭐ recommended for your own footage

Bring your **own** drone clip + flight log and run the full pipeline on it — **without committing
anything to the repo**. The cell below stages your files into a git-ignored folder (`data/<name>/`,
already covered by `.gitignore`) and **auto-generates the `mission.yaml` descriptor** for you, then
points `MISSION` at it. To run new data later, just change the paths and re-run — never a `git` commit.

**What DRISHTI needs from you**

| Input | Required? | What it is |
|-------|-----------|------------|
| **video** | ✅ **required** | your single-pass survey clip (`.mp4` / `.mov` / `.mkv`) |
| **telemetry** | ✅ **required** | GPS / flight log — `dji_srt` (DJI `.SRT`), `csv` (+ column map), `mavlink` (`.tlog` / `.bin`), or `exif` |
| intrinsics | optional | camera-intrinsics `.yaml` / `.json` (self-calibrated if omitted) |
| imu / baro | optional | inertial / barometer logs (tighten the trajectory when present) |
| rtk | optional | RTK/PPK-corrected track (better absolute accuracy) |
| check-points | optional | surveyed points (GeoJSON/CSV) → turns accuracy from **UNVALIDATED** into a measured number |

**How to give the cell each file** — for every path variable:
`"upload"` pops a file picker (Colab) · a real path uses that file in place · `""` skips an optional
(a blank **required** input is treated as `"upload"`). So you get an upload option for **each** input,
required or optional.

**Getting files onto the machine**
- **Colab** — leave `"upload"` and a **picker appears** for each such input (best for small/medium files).
  For a **big video**, mount Drive and paste the path instead:
  `from google.colab import drive; drive.mount('/content/drive')` → `VIDEO = "/content/drive/MyDrive/…/flight.mp4"`.
- **Kaggle** — there's no pop-up uploader. Click **＋ Add Input → Upload** (files land read-only under
  `/kaggle/input/<your-dataset>/`), or drag them into the `/kaggle/working` file browser — then paste the
  path(s) into the variables below. Uploading as a Kaggle *dataset* is the no-commit way to bring a large
  video to Kaggle.

Set the variables, run this cell, then run **§6** (6B on Kaggle's ~30 GB tier for aerial flights).

In [ ]:
# == IMPORT · Option E — UPLOAD YOUR OWN DATA (no repo commit) ==============================
# Edit this block. On Kaggle use real paths; the upload widget is Colab-only.
DATASET_NAME = "my_survey"
VIDEO = "upload"
TELEMETRY = "upload"
TELE_FORMAT = "dji_srt"  # dji_srt | csv | mavlink | exif
CSV_MAP = {"time": "timestamp", "lat": "latitude", "lon": "longitude",
           "alt": "abs_alt", "yaw": None, "pitch": None, "roll": None}
IMU = ""
BARO = ""
INTRINSICS = ""
RTK = ""
CHECK_POINTS = ""
CRS_MODE = "derive_from_gps"
CRS_EPSG = None
TARGET_FPS = 4.0
MAX_FRAMES = 0
TIME_OFFSET_S = 0.0
USE_UPLOAD_WIDGET = True

# Nothing below needs editing. Host flags come from the initial setup cell.
import os, shutil, subprocess
from pathlib import Path
import yaml

os.chdir(REPO_DIR)
STAGE_DIR = REPO_DIR / "data" / DATASET_NAME
STAGE_DIR.mkdir(parents=True, exist_ok=True)

def _pick(role, value, required):
    value = (value or "").strip()
    if value and value.lower() != "upload":
        path = Path(value).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"{role}: file not found: {path}")
        return str(path.resolve())
    if not required and not value:
        return None
    if not (ON_COLAB and USE_UPLOAD_WIDGET):
        raise ValueError(f"{role}: set an explicit file path. On Kaggle use Add Input, then /kaggle/input/...")
    from google.colab import files
    print("Upload", role)
    uploaded = files.upload()
    if not uploaded and not required:
        return None
    if len(uploaded) != 1:
        raise ValueError(f"Upload exactly one {role} file.")
    name = next(iter(uploaded))
    dest = STAGE_DIR / Path(name).name
    shutil.move(name, str(dest))
    return str(dest.resolve())

video_path = _pick("video", VIDEO, True)
tele_path = _pick("telemetry", TELEMETRY, True)
optional = {role: _pick(role, value, False) for role, value in
            (("imu", IMU), ("baro", BARO), ("intrinsics", INTRINSICS), ("rtk", RTK))}
check_points = _pick("check_points", CHECK_POINTS, False)
if CRS_MODE == "epsg" and not CRS_EPSG:
    raise ValueError("CRS_MODE='epsg' requires CRS_EPSG; use derive_from_gps for automatic selection.")
telemetry = {"format": TELE_FORMAT, "path": tele_path}
if TELE_FORMAT == "csv":
    telemetry["csv"] = dict(CSV_MAP)
descriptor = {
    "name": DATASET_NAME, "video": video_path, "telemetry": telemetry,
    "optional": optional, "crs": {"mode": CRS_MODE, "epsg": CRS_EPSG},
    "report": {"check_points": check_points},
}
mission_path = STAGE_DIR / f"{DATASET_NAME}.yaml"
mission_path.write_text(yaml.safe_dump(descriptor, sort_keys=False), encoding="utf-8")
subprocess.run([PY, "-c",
    "import sys; from drishti.config.loader import load_dataset; load_dataset(sys.argv[1]); print('VALID')",
    str(mission_path)], check=True)
MISSION = str(mission_path)
RESUME_BUNDLE = None
os.environ["DRISHTI_MISSION"] = MISSION
for key, value, default in (("target_fps", TARGET_FPS, 4.0), ("max_frames", MAX_FRAMES, 0),
                            ("time_offset_s", TIME_OFFSET_S, 0.0)):
    env_key = f"DRISHTI__ingest__{key}"
    if value == default:
        os.environ.pop(env_key, None)
    else:
        os.environ[env_key] = str(value)
print("MISSION =", MISSION)
print(mission_path.read_text())
print("Next: run 6B for aerial footage. Accuracy remains unvalidated without usable check-points.")

## 6 · Run the FULL pipeline (all 11 stages) — run **ONE** of the two blocks below

Both blocks run every stage `s0_ingest → … → report` on the **`balanced`** profile at the **same 5 cm
TSDF resolution** (`dense.tsdf_voxel_m=0.05`) and both use **exhaustive feature matching**
(`poses.matcher=exhaustive`). Exhaustive matching is essential here: the sample datasets are
**lawnmower-grid aerial surveys**, and the default *sequential* matcher only links consecutive frames,
so adjacent strips never loop-close — the trajectory drifts (~26 m on aukerman) and the mesh smears into
floating blocks. Verified locally on aukerman: trajectory RMSE **26.3 m → 6.5 m**, sparse points
**15.9k → 34.9k**, **75/75** frames registered.

They differ in **one** knob, `dense.depth_trunc_m` — the **maximum depth** the dense stage integrates.
This is not a free RAM dial: Open3D discards every pixel deeper than it, so `depth_trunc_m` **must exceed
your flying height (AGL)** or the ground is deleted.

| Block | Use on | `depth_trunc_m` | For |
|-------|--------|-----------------|-----|
| **6A · free tier** | Colab/Kaggle free T4 (**~13 GB RAM**) | **40 m** | **LOW-altitude** passes only — ground within ~40 m of the camera. |
| **6B · high-RAM (recommended)** | **Kaggle (free, ~30 GB)** or Colab **High-RAM** | **150 m** | **Aerial mapping** like the **aukerman sample** (~100–130 m AGL). Reaches the ground ~100 m below. |

> **Why aukerman needs 6B.** The aukerman flight is ~100–130 m above ground, so each camera sees the
> ground at ~100 m depth. 6A's 40 m clamp deletes it (→ the floating-fragment mesh you saw; DSM coverage
> 0.3 %). Keep the depth range appropriate to the flight. Use sufficient host RAM and tiled fusion;
> roughly 30 GB is the intended aerial setup, not a guarantee that every scene fits.

> **Honest caveats.**
> - Sizing `depth_trunc_m`: use ≈ **1.3 × your max AGL**. Too low deletes ground; far higher just
>   integrates unreliable long-range monocular depth and costs RAM.
> - `[exit -9]` means the process was killed; host-RAM exhaustion is a common cause. Identify the
>   last stage before tuning. S7 is tiled and S9/S10 stream large point products, but no universal
>   memory ceiling is guaranteed. The tile default is already **40 m**. For S7, a smaller tile such
>   as `--set dense.tile.tile_m=20` is a **fresh-run** tuning option. Resume accepts no settings
>   overrides; to apply changes set `RESUME_BUNDLE = None` and start a new run.
> - `--profile max` now selects the real CUDA/gsplat Gaussian + Apache DA3 path and Poisson meshing;
>   use a larger CUDA runtime for it. `balanced` remains the reproducible T4 path.

`DRISHTI_LOG_LEVEL=DEBUG` gives verbose logs, and the **entire console is tee'd** into
`logs/run_console.log`. The first run also fetches the RT-DETR and Depth-Anything-V2 Small checkpoints
(both Apache-2.0) into the model cache.

In [ ]:
# == RUN · Block 6A — LOW-ALTITUDE footage only. Aerial samples -> use 6B. ===================
# Run 6A OR 6B, not both. A 40 m depth limit discards ground farther from the camera.
import os, subprocess, sys, time
from pathlib import Path

os.environ["DRISHTI_LOG_LEVEL"] = "DEBUG"
os.environ["DRISHTI_RUNS_DIR"] = str(REPO_DIR / "runs")
FREE_DEPTH_TRUNC_M = 40
if RESUME_BUNDLE:
    bundle_path = Path(RESUME_BUNDLE).resolve()
    cmd = [PY, "-m", "drishti.cli", "resume", str(bundle_path)]
else:
    os.chdir(REPO_DIR)
    assert MISSION and Path(MISSION).is_file(), "Run an input cell in section 5 first."
    run_id = time.strftime("colab-free-%Y%m%d-%H%M%S")
    bundle_path = REPO_DIR / "runs" / run_id
    cmd = [PY, "-m", "drishti.cli", "run", "--dataset", MISSION, "--profile", "balanced",
           "--run-id", run_id, "--output-root", str(REPO_DIR / "runs"),
           "--set", "poses.matcher=exhaustive", "--set", f"dense.depth_trunc_m={FREE_DEPTH_TRUNC_M}"]
BUNDLE = str(bundle_path)
logfile = bundle_path / "logs" / "run_console.log"
logfile.parent.mkdir(parents=True, exist_ok=True)
print(">>", " ".join(cmd))
print("Console tee ->", logfile)
with open(logfile, "a", encoding="utf-8", buffering=1) as lf:
    lf.write(f"\n=== {time.strftime('%Y-%m-%d %H:%M:%S')} | {' '.join(cmd)} ===\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        lf.write(line)
    proc.wait()
print(f"\n[exit {proc.returncode}] bundle: {BUNDLE}")
if proc.returncode in (-9, 137):
    print("Process killed; inspect the last stage and host RAM. Keep the depth limit appropriate to the flight.")
if proc.returncode:
    print("Fix the cause, set RESUME_BUNDLE = BUNDLE and retry with saved settings.")
    print("For different settings start a fresh run. Previous console attempts are preserved.")

In [ ]:
# == RUN · Block 6B — HIGH-RAM (>= 30 GB): full quality + exhaustive matching (RECOMMENDED) =====
# Run this on Kaggle (free, ~30 GB RAM) or a Colab High-RAM runtime. This is the RIGHT block for the
# aukerman sample and any aerial-mapping flight: (1) exhaustive matching so the lawnmower strips loop-
# close (sequential drifts ~26 m -> smeared mesh), and (2) dense.depth_trunc_m=150 so the dense stage
# reaches the GROUND ~100 m below the ~100-130 m AGL cameras (a lower clamp deletes it -> floating
# blocks). 5 cm voxel unchanged -- full resolution. >>> Run 6A OR 6B, not both; 6B does not need 6A. <<<
import os, subprocess, sys, time
from pathlib import Path

os.environ["DRISHTI_LOG_LEVEL"] = "DEBUG"
os.environ["DRISHTI_RUNS_DIR"] = str(REPO_DIR / "runs")

# NOTE on resume: `drishti resume` reuses the config recorded in the bundle's manifest, so resuming a
# 6A bundle keeps 6A's near-field clamp. To get 6B's ground-reaching 150 m range + exhaustive matching,
# start a FRESH run (RESUME_BUNDLE = None in §5) rather than resuming a 6A bundle.
if RESUME_BUNDLE:
    bundle_path = Path(RESUME_BUNDLE).resolve()
    cmd = [PY, "-m", "drishti.cli", "resume", str(bundle_path)]
else:
    os.chdir(REPO_DIR)
    assert MISSION and Path(MISSION).is_file(), "no MISSION — run an import cell in §5 first."
    run_id = time.strftime("colab-full-%Y%m%d-%H%M%S")
    bundle_path = REPO_DIR / "runs" / run_id
    (bundle_path / "logs").mkdir(parents=True, exist_ok=True)
    # exhaustive matching (cross-strip loop closure) + depth_trunc=150 (reach the ground ~100 m below
    # the ~100-130 m AGL flight); 5 cm voxel unchanged. Both are required -- see the header.
    cmd = [PY, "-m", "drishti.cli", "run", "--dataset", MISSION, "--profile", "balanced", "--run-id", run_id,
           "--output-root", str(REPO_DIR / "runs"),
           "--set", "poses.matcher=exhaustive", "--set", "dense.depth_trunc_m=150"]

BUNDLE = str(bundle_path)
logfile = bundle_path / "logs" / "run_console.log"
logfile.parent.mkdir(parents=True, exist_ok=True)
print(">>", " ".join(cmd))
print("   console tee ->", logfile, "\n")

with open(logfile, "a", encoding="utf-8", buffering=1) as lf:
    lf.write(f"\n=== {time.strftime('%Y-%m-%d %H:%M:%S')} | {' '.join(cmd)} ===\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        sys.stdout.write(line)
        lf.write(line)
    proc.wait()

BUNDLE = str(bundle_path)
print(f"\n[exit {proc.returncode}]  bundle: {BUNDLE}")
if proc.returncode in (-9, 137):
    print("\nThe process was killed; host-RAM exhaustion is a common cause. Check the last stage and\n"
          "  resource usage. S7 is tiled; S9/S10 stream point products, but a scene may still exceed RAM.\n"
          "  The default tile is already 40 m. If S7 is the bottleneck, try a smaller tile (e.g. 20 m)\n"
          "  in a FRESH run: edit the run command and set RESUME_BUNDLE = None. Resume takes no --set.\n"
          "  Keep depth_trunc appropriate to the flight; 6A can delete high-altitude ground.\n"
          "  For unchanged settings, set RESUME_BUNDLE = BUNDLE and retry; console attempts are preserved.\n"
          "  Address the cause first. Completed fresh stages are skipped; the interrupted stage restarts.")
elif proc.returncode != 0:
    print("A stage failed above — that is the honest, fail-loud behaviour. Inspect it in §7, fix the "
          "cause, then set  RESUME_BUNDLE = BUNDLE  and re-run; completed stages are skipped.")

## 7 · Inspect + verify (the ground truth)

`inspect` prints the manifest — per-stage status, environment, measured wall-time, stage-specific
confidence/evidence signals, and degradation. Confidence is not a calibrated accuracy guarantee.
`verify` checks output hashes against the manifest; it tests file integrity, not geometry accuracy.

In [ ]:
!drishti inspect "$BUNDLE"
!drishti verify "$BUNDLE"

## 8 · Logs & report

Everything the run recorded: the tee'd console log, the per-stage manifest (above), and the accuracy
report the final stage wrote.

In [ ]:
import json
from pathlib import Path
B = Path(BUNDLE)

print("=== logs/ ===")
for p in sorted((B / "logs").glob("*")):
    print(f"  {p.name:24} {p.stat().st_size:>10,} B")

rc = B / "logs" / "run_console.log"
if rc.is_file():
    print("\n=== last 25 lines of run_console.log ===")
    print("".join(rc.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)[-25:]))

print("=== report/ ===")
for p in sorted((B / "report").glob("*")):
    print("  ", p.name)

rj = B / "report" / "report.json"
if rj.is_file():
    r = json.loads(rj.read_text(encoding="utf-8"))
    print("\n=== report.json (summary) ===")
    for k in ("drishti_version", "total_wall_seconds", "n_failed", "n_degraded"):
        if k in r:
            print(f"  {k}: {r[k]}")
    if "accuracy" in r:
        print("  accuracy:", json.dumps(r["accuracy"]))
    print("\nOpen report/report.html locally for the full formatted report.")

## 9 · Export EVERYTHING — download the whole bundle (incl. logs)

Two archives:
- **full** — the complete run directory, including existing stage outputs, manifest, logs, and report.
  Use this for output verification and resume. Original video/telemetry, code, and model caches normally
  live outside this folder: preserve them separately for reproduction.
- **slim** — `s10_export`, report, logs, and manifest. For viewing/sharing; omitted intermediates mean
  whole-bundle verification will report missing artifacts, and this is not a resume package.

Archives are created beside the bundle, normally `/kaggle/working/drishti/runs/` on Kaggle. Download
from the file browser before ending the session. Archiving needs additional disk space. The commented
Google Drive option below applies to Colab. See [the guide](KAGGLE-GUIDE.md#8-download-and-preserve-results-notebook-section-9).

In [ ]:
import shutil, zipfile
from pathlib import Path
B = Path(BUNDLE)
rid = B.name

# --- full bundle ---
full_zip = shutil.make_archive(str(B.parent / f"{rid}_full"), "zip", root_dir=B.parent, base_dir=rid)
print("full :", full_zip, f"({Path(full_zip).stat().st_size / 1e6:.1f} MB)")

# --- slim: deliverables + report + logs + manifest ---
slim_zip = str(B.parent / f"{rid}_slim.zip")
with zipfile.ZipFile(slim_zip, "w", zipfile.ZIP_DEFLATED) as z:
    if (B / "manifest.json").is_file():
        z.write(B / "manifest.json", f"{rid}/manifest.json")
    for sub in ["report", "logs", "s10_export"]:
        for p in (B / sub).rglob("*"):
            if p.is_file():
                z.write(p, f"{rid}/{p.relative_to(B)}")
print("slim :", slim_zip, f"({Path(slim_zip).stat().st_size / 1e6:.1f} MB)")

# --- download: only use the Colab widget when actually on Colab ---
if ON_COLAB:
    from google.colab import files
    files.download(slim_zip)      # small first; swap to full_zip for everything
else:
    print("Download the archives from the Kaggle file/output browser at the paths printed above.")

# --- or copy to Drive for large bundles (Colab only) ---
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy2(full_zip, "/content/drive/MyDrive/")

## 10 · Round-trip back to your machine

Follow [the local-viewing guide](KAGGLE-GUIDE.md#10-view-results-on-your-computer-notebook-section-10)
for exact Windows and Linux/macOS commands, virtual-environment setup, and the two server terminals.

The simplest route is to extract the archive, open `report/report.html`, and open the successful
GLB/glTF/OBJ/PLY exports in Blender or another mesh viewer; use QGIS for the GeoTIFFs.

The DRISHTI web viewer uses the light `server` extra plus Node/npm. Its model panel requires a
successful **3D Tiles** export; it does not render GLB as a fallback.

Use `inspect` and `verify` on the full bundle for status and integrity. Resume is a processing
command that can require original inputs and heavy dependencies, not a command for viewing a
finished model. Slim archives intentionally omit intermediate artifacts, so full verification
will report missing files on them.